# Your first policy bundle

This notebook takes you from an empty directory to one compiled, inspected
policy bundle, running the real `toolgraph` command at every step. It covers
the same ground as the [Beginner Guide](../beginner-guide.md), with the outputs
in front of you and two hands-on experiments the guide only describes.

**What Toolgraph does.** It crawls the tools that MCP servers expose, combines
them with what an operator has written down (who may call which tool, what each
tool reads or writes, which policy governs which resource), and explains every
`Agent -> Tool -> Resource -> Policy` path it finds. The result is a JSON
bundle that a gateway such as memtomem-stm uses to hide or block tools.

**When you need it.** You run agents against MCP servers and want a reviewable
answer to "which tools may this agent use, and why not the others?" before a
call happens.

**When you don't.** Toolgraph does not block calls. It advises and
compiles. If you only want to see what a server exposes, the server's own tool
listing is enough.

**How to run it.** Use a Jupyter kernel whose environment has Toolgraph
installed:

```bash
pip install "toolgraph[ladybug]" ipykernel
```

Then choose **Restart Kernel and Run All Cells**. Each cell depends on the ones
above it. Everything happens in a temporary directory, and the last cell deletes
it. No Docker, Node.js, network access or repository clone is needed.

## Setup

The cell below finds the `toolgraph` command installed in this kernel's Python
environment, creates a temporary sandbox, and defines `run(...)`, which runs one
`toolgraph` command inside the sandbox and prints what it printed.

A kernel inherits the environment of the shell that started it. A
`TOOLGRAPH_BACKEND` left over from another project would make every command
below stop with `conflicting backend configuration`. The helper therefore drops
`TOOLGRAPH_*`, `NEO4J_*` and Python path overrides before running anything.
Paths printed by the commands show the sandbox as `<sandbox>`.

In [1]:
import hashlib
import json
import os
import shutil
import subprocess
import sysconfig
import tempfile
from pathlib import Path

# Console scripts live here: bin in a venv, Scripts on Windows and in Conda.
_SCRIPTS = Path(sysconfig.get_path("scripts"))
TOOLGRAPH = next(
    (path for path in (_SCRIPTS / "toolgraph", _SCRIPTS / "toolgraph.exe") if path.exists()),
    None,
)
if TOOLGRAPH is None:
    raise RuntimeError(
        "No toolgraph command in this kernel's Python environment. Install it into the "
        'kernel\'s environment: pip install "toolgraph[ladybug]"'
    )

_DROPPED_PREFIXES = ("TOOLGRAPH_", "NEO4J_")
_DROPPED_NAMES = {"PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "PYTHONUSERBASE"}
ENV = {
    key: value
    for key, value in os.environ.items()
    if not key.startswith(_DROPPED_PREFIXES) and key not in _DROPPED_NAMES
}

_created = Path(tempfile.mkdtemp(prefix="toolgraph-notebook."))
SANDBOX = _created.resolve()
QUICKSTART = SANDBOX / "quickstart"
# Longest first, so a resolved /private/var/... is replaced before /var/...
_SHOWN_AS_SANDBOX = sorted({str(SANDBOX), str(_created)}, key=len, reverse=True)


def run(*args, cwd=QUICKSTART, check=True):
    """Run `toolgraph <args>` inside the sandbox and print its output."""
    done = subprocess.run([str(TOOLGRAPH), *args], cwd=cwd, env=ENV, capture_output=True, text=True)
    shown = done.stdout + done.stderr
    for path in _SHOWN_AS_SANDBOX:
        shown = shown.replace(path, "<sandbox>")
    print(shown, end="")
    if check and done.returncode != 0:
        raise RuntimeError(f"toolgraph {args[0]} exited with {done.returncode}")
    return done


def run_json(*args):
    """Run a command that prints JSON, print it, and return the parsed value."""
    return json.loads(run(*args).stdout)


run("--version", cwd=SANDBOX);

toolgraph 0.2.0


## 1. Create the quickstart and initialize it

`toolgraph example init` copies a self-contained example into a new directory.
It refuses a directory that already exists, so it never overwrites your files.

In [2]:
run("example", "init", "quickstart", cwd=SANDBOX)
sorted(path.name for path in QUICKSTART.iterdir())

Created Toolgraph quickstart at <sandbox>/quickstart
Next: cd <sandbox>/quickstart && toolgraph init


['README.md', 'governance.yaml', 'policy_gateway_server.py', 'servers.yaml']

The four files are everything this walkthrough needs:

- `policy_gateway_server.py`: a tiny MCP server with two demo tools. It runs
  offline over stdio.
- `servers.yaml`: tells `crawl` how to start that server.
- `governance.yaml`: the operator-authored half: grants, data flows and a
  policy.
- `README.md`: the same steps as a shell walkthrough.

`toolgraph init` creates `.toolgraph/config.json` and a local Ladybug database
in the current directory. Ladybug is an embedded graph database, so there is no
server to run.

In [3]:
run("init");

Initialized backend=ladybug config=<sandbox>/quickstart/.toolgraph/config.json


`toolgraph doctor` checks the config file, the database connection, the schema
and the graph state. Run it whenever something looks wrong; it is the fastest
way to find out which of the four is the problem.

In [4]:
run("doctor");

Toolgraph Doctor Diagnosis:
  [PASS] Configuration: Valid config at .toolgraph/config.json (mode 0o600)
         - path: .toolgraph/config.json
         - exists: True
         - valid_json: True
         - permissions_ok: True
         - permissions: 0o600
         - healthy: True
         - detail: ok
         - backend_source: config-file
  [PASS] Connectivity: Backend 'ladybug' connected and responsive
         - backend: ladybug
  [PASS] Schema Integrity: Schema version 1 intact (13 tables present)
         - backend: ladybug
         - schema_version: 1
         - expected_version: 1
         - version_ok: True
         - live_tables: ['Agent', 'CAN_CALL', 'EXPOSES', 'ExpectedException', 'GOVERNED_BY', 'GraphMeta', 'MCPServer', 'PROVIDES', 'Policy', 'READS', 'Resource', 'Tool', 'WRITES']
         - missing_tables: []
         - live_constraints: []
         - missing_constraints: []
         - healthy: True
  [PASS] Graph State: Instance f54ea312-9c5c-4e8c-9b73-ed88e7cb258c, gene

All four checks report `PASS` and the last line is `Overall status: HEALTHY`.
`backend_source: config-file` says the backend choice came from the file `init`
just wrote. Any `FAIL` makes `doctor` exit 1, and the
[troubleshooting section](../beginner-guide.md#troubleshooting) of the guide
explains each one.

## 2. Crawl the MCP server

Crawling starts each server in `servers.yaml`, asks it which tools and
resources it exposes, and records the answer in the graph. Crawled facts are
what the server says about itself. Nothing here is authored by you yet.

In [5]:
run("crawl", "--servers", "servers.yaml");

Crawling 1 server(s) from servers.yaml ...
  [ok] policy-gateway (2 tools, 1 resources)
Graph now: {'mcpserver': 1, 'tool': 2, 'resource': 1, 'exposes': 2, 'provides': 1}


The server exposes two tools:

- `policy-gateway::read_note`: read-only and idempotent.
- `policy-gateway::publish_note`: a mutating tool.

Tools are named `server::tool`. This notebook always uses that qualified name,
which stays unambiguous when two servers expose a tool with the same name.

## 3. Load the authored policy

Crawling tells Toolgraph what exists. `governance.yaml` tells it what is
allowed and what each tool touches, which a server cannot be trusted to say
about itself. Read it once:

In [6]:
print((QUICKSTART / "governance.yaml").read_text())

# The operator-authored half of the quickstart graph. Crawling discovers what
# tools exist; this file asserts who may call them and what they touch.
#
# Every authored edge carries `provenance`. `source: operator_asserted` means a
# human claimed it (not a crawled fact), and `evidence` points at where the
# claim can be checked. `toolgraph unbacked-edges` lists edges missing that
# pointer, so a real deployment should keep this section filled in.
#
# Read the pointers here for what they are. The demo tools return strings and
# touch no file at all, so the READS/WRITES entries are synthetic effects that
# exist to make the DENY path reachable; their evidence cites the fixture that
# defines them. An empty `unbacked-edges` result means every edge has a
# pointer, never that a claim is true. Checking the claim is the reviewer's
# job, which is the whole reason the pointer is required.

agents:
  - vibe-coder

policies:
  - id: draft-publish-deny
    effect: DENY
    scope: demo-drafts
  

In short: `vibe-coder` is granted both tools, `publish_note` writes to
`file:///demo/drafts`, and that resource is governed by a `DENY` policy. Every
authored edge carries `provenance`: who asserted it and where the claim can be
checked.

Preview the load first. `--dry-run` checks the manifest against the graph and
prints what would change without writing anything. `--strict-drift` turns a
reference to a tool the crawl did not find into an error instead of a warning.

In [7]:
run("ingest-manifest", "--governance", "governance.yaml", "--strict-drift", "--dry-run");

DRY-RUN: manifest at governance.yaml would apply cleanly
DRY-RUN RESULT: VALID (would apply cleanly)
  Graph state token: {'instance_id': 'f54ea312-9c5c-4e8c-9b73-ed88e7cb258c', 'generation': 1}
  Delta summary:
    Agents: +1 / -0
    Policies: +1 / -0 / ~0
    Grants: +2 / -0 / ~0 (granted_by)
    Data access: +2 / -0
    Governed by: +1 / -0
    Exceptions: +0 / -0 / ~0
    Provenance modified: 0
    Resources: +1 created / -0 pruned (final degree 0) / 0 retained crawled
    Tools: -0 pruned (final degree 0) / 0 retained crawled


The delta matches the file: one agent, one policy, two grants, two data-access
edges and one policy binding. A manifest that would be rejected prints
`DRY-RUN REJECTED` with the reason and exits 1. The usual cause is a misspelled
tool name. Load it for real:

In [8]:
run("ingest-manifest", "--governance", "governance.yaml", "--strict-drift");

Ingested governance from governance.yaml
Governance now: {'agent': 1, 'policy': 1, 'can_call': 2, 'reads': 1, 'writes': 1, 'governed_by': 1, 'expected_exception': 0}


`unbacked-edges` lists authored edges that have no evidence pointer: data
flows (READS/WRITES) and policy bindings (GOVERNED_BY). Every edge in this
manifest has one, so the list is empty:

In [9]:
unbacked = run_json("unbacked-edges")

[]


An empty list means every edge has a pointer, not that the claims behind them
are true. The demo tools return strings and touch no file, so their
READS/WRITES entries are synthetic, and their evidence says so. Checking a
pointer is a reviewer's job. Requiring one is what makes that job possible.

## 4. Ask for the decision

`eligible-tools` answers the question the bundle will encode: of these
candidate tools, which may `vibe-coder` use under the `review` profile?

In [10]:
decision = run_json(
    "eligible-tools",
    "vibe-coder",
    "policy-gateway::read_note",
    "policy-gateway::publish_note",
    "--profile",
    "review",
)

{
  "agent": "vibe-coder",
  "agent_found": true,
  "profile": "review",
  "eligible": [
    "policy-gateway::read_note"
  ],
  "rejected": [
    {
      "candidate": "policy-gateway::publish_note",
      "tool_key": "policy-gateway::publish_note",
      "reason": "DENY_VIOLATION",
      "paths": [
        "(publish_note) -WRITES-> (file:///demo/drafts) -GOVERNED_BY-> (draft-publish-deny:DENY)"
      ]
    }
  ]
}


`read_note` is eligible. `publish_note` is rejected with `DENY_VIOLATION`, and
`paths` shows why: `publish_note` writes to `file:///demo/drafts`, which is
governed by `draft-publish-deny`. Toolgraph is reporting the decision here. It
has not intercepted any call.

`selection-explain` shows every reason behind a single decision, including the
ones that did not change the outcome:

In [11]:
explained = run_json(
    "selection-explain", "vibe-coder", "policy-gateway::publish_note", "--profile", "review"
)

{
  "agent": "vibe-coder",
  "tool": "policy-gateway::publish_note",
  "tool_key": "policy-gateway::publish_note",
  "agent_found": true,
  "found": true,
  "profile": "review",
  "decision": "rejected",
  "risk_score": 1.0,
  "reasons": [
    "agent has CAN_CALL grant",
    "DENY path [violation]: (publish_note) -WRITES-> (file:///demo/drafts) -GOVERNED_BY-> (draft-publish-deny:DENY)",
    "tool is currently exposed by a crawled server",
    "data-flow effects are authored (READS/WRITES present)",
    "all load-bearing edges carry evidence",
    "server self-claim (crawled/medium): destructive_hint=True"
  ]
}


The grant exists and the tool is live, but the DENY path wins. The last reason
is the server's own claim that the tool is destructive. Toolgraph records it as
`crawled/medium`: a crawled fact, but an unverified one, so it never gets the
`high` confidence of the crawled tool listing itself.

## 5. Compile the bundle

`policy compile` writes the decisions for one agent and profile to a file a
gateway can load.

In [12]:
compiled = run_json(
    "policy",
    "compile",
    "--agent",
    "vibe-coder",
    "--profile",
    "review",
    "--output",
    ".toolgraph/policy-bundle.json",
)

{
  "kind": "toolgraph.policy-bundle",
  "output": ".toolgraph/policy-bundle.json",
  "bundle_digest": "d6c4d322d384091f622fbd14f49d4912650f485c686d7a2f8d46ddca337ca090",
  "graph_state": {
    "instance_id": "f54ea312-9c5c-4e8c-9b73-ed88e7cb258c",
    "generation": 2
  },
  "agent": "vibe-coder",
  "profile": "review",
  "eligible": 1,
  "rejected": 1
}


That output describes the bundle: where it was written, its byte digest, the
graph state it was compiled from, and the eligible/rejected counts. The digest
and instance id differ on every run. Open the bundle itself to see what a
gateway reads. `bundle_digest` is the SHA-256 of the file's exact bytes, so
anyone holding the file can check that it is the one you compiled:

In [13]:
raw = (QUICKSTART / ".toolgraph" / "policy-bundle.json").read_bytes()
bundle = json.loads(raw)
print("digest matches:", hashlib.sha256(raw).hexdigest() == compiled["bundle_digest"])
print("graph state matches:", bundle["graph_state"] == compiled["graph_state"])
print()
for tool in bundle["tools"]:
    print(f"{tool['tool_key']:32} {tool['decision']:9} {tool.get('reason', '')}")

digest matches: True
graph state matches: True

policy-gateway::publish_note     rejected  DENY_VIOLATION
policy-gateway::read_note        eligible  


Each entry also carries a `tool_contract_digest`, a hash of the tool's crawled
contract: its server, name, description, input schema and annotation hints. If
the server changes the tool, the digest no longer matches the bundle.

Start with `review`: the reference gateway keeps rejected tools visible and
records the calls it would have blocked. After inspecting those decisions,
compile a matching `strict` bundle to hide and block rejected tools.

## 6. Experiment: remove one evidence pointer

This is the experiment the guide asks you to do by hand. The cell removes the
`provenance` block from the `publish_note` entry under `data_access`, keeping
the original bytes so the next cells can put them back.

In [14]:
GOVERNANCE = QUICKSTART / "governance.yaml"
original = GOVERNANCE.read_bytes()
block = (
    "    provenance:\n"
    "      source: operator_asserted\n"
    "      confidence: high\n"
    '      evidence: "policy_gateway_server.py:publish_note (synthetic demo effect)"\n'
)
text = original.decode()
assert text.count(block) == 1, "expected exactly one publish_note provenance block"
GOVERNANCE.write_text(text.replace(block, ""))
run("ingest-manifest", "--governance", "governance.yaml", "--strict-drift", "--dry-run");

DRY-RUN: manifest at governance.yaml would apply cleanly
DRY-RUN RESULT: VALID (would apply cleanly)
  Graph state token: {'instance_id': 'f54ea312-9c5c-4e8c-9b73-ed88e7cb258c', 'generation': 2}
  Delta summary:
    Agents: +0 / -0
    Policies: +0 / -0 / ~0
    Grants: +0 / -0 / ~0 (granted_by)
    Data access: +0 / -0
    Governed by: +0 / -0
    Exceptions: +0 / -0 / ~0
    Provenance modified: 1
    Resources: +0 created / -0 pruned (final degree 0) / 0 retained crawled
    Tools: -0 pruned (final degree 0) / 0 retained crawled


The dry run reports `Provenance modified: 1` and nothing else, because the edge
itself is unchanged. Only its evidence is gone. Load it and audit again:

In [15]:
run("ingest-manifest", "--governance", "governance.yaml", "--strict-drift")
unbacked = run_json("unbacked-edges")

Ingested governance from governance.yaml
Governance now: {'agent': 1, 'policy': 1, 'can_call': 2, 'reads': 1, 'writes': 1, 'governed_by': 1, 'expected_exception': 0}


[
  {
    "edge_type": "WRITES",
    "src_label": "Tool",
    "src": "policy-gateway::publish_note",
    "dst_label": "Resource",
    "dst": "file:///demo/drafts",
    "source": "operator_asserted",
    "confidence": "high"
  }
]


`unbacked-edges` now names exactly that edge: `publish_note` `WRITES`
`file:///demo/drafts`. It still says who asserted it (`operator_asserted`,
`high`). What is missing is the evidence pointer a reviewer would follow. In a
real deployment, this is the list to work through before trusting a bundle.

Grants are left out by default because a grant declares intent rather than a
fact about what a tool touches. `unbacked-edges --include-grants` audits them
too.

Put the original file back and reload it. The rest of the notebook assumes the
original manifest:

In [16]:
GOVERNANCE.write_bytes(original)
run("ingest-manifest", "--governance", "governance.yaml", "--strict-drift")
unbacked = run_json("unbacked-edges")

Ingested governance from governance.yaml
Governance now: {'agent': 1, 'policy': 1, 'can_call': 2, 'reads': 1, 'writes': 1, 'governed_by': 1, 'expected_exception': 0}


[]


## 7. The audits

These commands answer the questions a reviewer asks about the whole graph
rather than one decision.

`unsafe-tools` lists the tools an agent can call that touch a DENY-governed
resource. On this fixture it returns a row on purpose: naming that reach is
what it is for. Add `--fail-on-violation` to make it exit 1 in CI.

In [17]:
unsafe = run_json("unsafe-tools", "vibe-coder")

[
  {
    "tool_key": "policy-gateway::publish_note",
    "tool": "publish_note",
    "server": "policy-gateway",
    "via": "resource",
    "mode": "WRITES",
    "resource": "file:///demo/drafts",
    "policy": "draft-publish-deny",
    "path": "(publish_note) -WRITES-> (file:///demo/drafts) -GOVERNED_BY-> (draft-publish-deny:DENY)",
    "provenance": {
      "source": "operator_asserted",
      "confidence": "high",
      "evidence": "policy_gateway_server.py:publish_note (synthetic demo effect)"
    },
    "policy_provenance": {
      "source": "operator_asserted",
      "confidence": "high",
      "evidence": "governance.yaml:draft-policy"
    },
    "classification": "violation"
  }
]


`blast-radius` works from the other end: given a policy, which agents and tools
reach the resources it governs?

In [18]:
blast = run_json("blast-radius", "draft-publish-deny")

{
  "kind": "policy",
  "node": "draft-publish-deny",
  "found": true,
  "impacted": [
    {
      "via": "resource",
      "resource": "file:///demo/drafts",
      "agent": "vibe-coder",
      "tool_key": "policy-gateway::publish_note",
      "tool": "publish_note",
      "mode": "WRITES",
      "path": "(vibe-coder) -CAN_CALL-> (publish_note) -WRITES-> (file:///demo/drafts)",
      "provenance": {
        "source": "operator_asserted",
        "confidence": "high",
        "evidence": "policy_gateway_server.py:publish_note (synthetic demo effect)"
      }
    }
  ]
}


Two audits come back empty here, and in this case empty is the correct answer:

- `unmapped-tools`: granted tools whose READS/WRITES are not authored yet.
  Toolgraph cannot reason about their data flow until they are.
- `drift`: tools with authored governance that the latest crawl no longer
  exposes. Re-crawl and re-ingest after fixing them.

In [19]:
unmapped = run_json("unmapped-tools")
drift = run_json("drift")

[]


[]


## Apply it to your own project

- [ ] List your MCP servers in a `servers.yaml`. Give every server an explicit,
      unique `name`. That name becomes the `server::` half of every tool key.
- [ ] Run `toolgraph crawl` and read the tool list before writing any policy.
- [ ] Write `governance.yaml`: agents, grants, data access and policies. Give
      every edge `provenance` that points at something a reviewer can open.
- [ ] Use `ingest-manifest --dry-run` before every real load, and keep
      `--strict-drift` on.
- [ ] Run `unbacked-edges`, `unmapped-tools` and `drift` until each one is
      empty or explained.
- [ ] Compile a `review` bundle first. Move to `strict` once the would-block
      decisions look right.
- [ ] Recompile after every crawl or manifest change. A bundle records the
      graph state it came from, so a stale one is detectable.

## Common pitfalls

- **Running a command outside the project directory.** Toolgraph reads
  `.toolgraph/config.json` from the current directory. Elsewhere, commands
  stop with `no backend configured` and `doctor` shows `backend_source: none`.
- **A leftover `TOOLGRAPH_BACKEND` or `TOOLGRAPH_DB_PATH`** in your shell or
  `.env` that disagrees with the config file. Commands stop with
  `conflicting backend configuration` and name the variable.
- **Expecting Toolgraph to block a call.** A `DENY` takes effect only when a
  gateway loads the bundle.

## Where next

- Enforce the bundle with memtomem-stm: see
  [section 6 of the guide](../beginner-guide.md#6-optional-enforce-it-with-memtomem-stm).
- Share a graph between processes or operators with Neo4j: see
  [Shared or fleet operation](../beginner-guide.md#shared-or-fleet-operation).

## Clean up

This deletes the sandbox, including the database and the bundle you compiled.
Rerun the notebook from the top to recreate them.

In [20]:
assert SANDBOX.name.startswith("toolgraph-notebook.")
shutil.rmtree(SANDBOX)
print("Removed <sandbox>")

Removed <sandbox>
